In [1]:
!pip install opencv-python
!pip install pillow
# Install dependencies
!pip install opencv-python-headless pillow

# Download Hindi font from a reliable source
#!wget -O lohit-devanagari.ttf https://raw.githubusercontent.com/JulienMa/GoogleFonts-ttf/master/ofl/lohitdevanagari/Lohit-Devanagari.ttf



In [2]:
import os
from openai import OpenAI
client = OpenAI(api_key="sk-proj-xxxxxxxxxxxxxxx")


# 1️⃣ Extract file name (without .mp4)
def get_filename(path):
    return os.path.splitext(os.path.basename(path))[0]


# Helper → enforce strict character limit
def enforce_limit(text, limit):
    text = text.strip()
    if len(text) > limit:
        return text[:limit].strip()
    return text


# 2️⃣ Hindi headline (STRICT ≤ 38 chars)
def rewrite_hindi(title):
    prompt = (
        f"Rewrite this into a VERY eye-catching Hindi news headline. "
        f"STRICT LIMIT: under 25 characters. "
        f"Do NOT exceed 30 characters. Under 25 characters with respectable language"
        f"Do NOT add quotes. "
        f"Rewrite this in Hindi. DO NOT use any emojis, symbols, or special characters or sensationaly disrespectful words "
        f"Use only pure easy Hindi text. TITLE: {title}"
        f"TITLE: {title}"
    )

    res = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=30,
        messages=[{"role": "assistant", "content": prompt}]
    )

    output = res.choices[0].message.content.strip()
    return enforce_limit(output, 38)


# 3️⃣ English thumbnail text (STRICT ≤ 26 chars)
def rewrite_english(title):
    prompt = (
        f"Rewrite this into a Respectable non sensational easy English for YT thumbnail headline. "
        f"STRICT LIMIT: Under 20 characters. "
        f"No quotes, no extra punctuation. Pick sentiment and name focused"
        f"TITLE: {title}"
    )

    res = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=18,
        messages=[{"role": "assistant", "content": prompt}]
    )

    output = res.choices[0].message.content.strip()
    return enforce_limit(output, 22)



# ▶ Example usage
file_title = get_filename("/content/content/India’s New Labour Codes ｜ Modi Govt Simplifies 29 Old Laws [RKEiPnH8dlg].mp4")

hindi = rewrite_hindi(file_title)
english = rewrite_english(file_title)

print("Hindi:", hindi, len(hindi))
print("English:", english, len(english))

Hindi: भारत के नए श्रम कोड: 29 कानून सरल 33
English: India's Labour Reform 21


In [6]:
from PIL import Image, ImageDraw, ImageFont
import cv2
import numpy as np

video_path = "/content/India’s New Labour Codes ｜ Modi Govt Simplifies 29 Old Laws [RKEiPnH8dlg].mp4"

# Extract frames
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

def extract_frame(sec):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fps * sec))
    ret, frame = cap.read()
    if not ret:
        return None
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)

frame1 = extract_frame(1)
frame2 = extract_frame(13)
cap.release()

# Resize frames
frame_width, frame_height = 320, 180
frame1 = frame1.resize((frame_width, frame_height))
frame2 = frame2.resize((frame_width, frame_height))

# Final thumbnail canvas
top_bar_height = 80
mid_bar_height = 80
final_width = frame_width * 2
final_height = top_bar_height + mid_bar_height + frame_height
thumbnail = Image.new("RGB", (final_width, final_height), color="white")
draw = ImageDraw.Draw(thumbnail)

# Load fonts
font_en = ImageFont.truetype("/content/dejavu-sans-bold.ttf", 36)
font_hi = ImageFont.truetype("/content/NotoSansDevanagari-Regular.ttf", 40)


# === Top red bar with English text ===
draw.rectangle([0, 0, final_width, top_bar_height], fill="red")
en_text = english #"Operation Sindoor Sansad" # 24 characters
bbox = draw.textbbox((0, 0), en_text, font=font_en)
text_w = bbox[2] - bbox[0]
text_h = bbox[3] - bbox[1]
draw.text(((final_width - text_w) // 2, (top_bar_height - text_h) // 2), en_text, fill="white", font=font_en)

# === White bar with Hindi text ===
hi_text = hindi #"राजनाथ सिंह संसद में बोल दिया ये बात .." # 39 characters
bbox = draw.textbbox((0, 0), hi_text, font=font_hi)
text_w = bbox[2] - bbox[0]
text_h = bbox[3] - bbox[1]
draw.text(((final_width - text_w) // 2, top_bar_height + (mid_bar_height - text_h) // 2), hi_text, fill="black", font=font_hi)

# Paste frames
thumbnail.paste(frame1, (0, top_bar_height + mid_bar_height))
thumbnail.paste(frame2, (frame_width, top_bar_height + mid_bar_height))

# Add logo
import requests
from io import BytesIO

logo_url = 'https://samachartimeshindi.com/images/logos/logo-f.png'
resp = requests.get(logo_url)
logo = Image.open(BytesIO(resp.content)).convert('RGBA')

# Resize logo
scale = 150
wpercent = scale / float(logo.width)
hsize = int(float(logo.height) * wpercent)
logo = logo.resize((scale, hsize), Image.LANCZOS)

# Paste bottom-right
thumbnail.paste(
    logo,
    (final_width - scale - 20, final_height - hsize - 20),
    logo
)


# Save
output_path = "final_thumbnail.jpg"
thumbnail.save(output_path)
thumbnail.show()
